# Agentic Design Patterns

A visual, runnable guide to nine control-flow patterns for building reliable LLM agents with LangGraph.
Each section pairs an architecture diagram with a compact example you can run from this notebook.

**How to read this notebook**

| Section | What you get |
| --- | --- |
| 1. Why agentic design patterns | The problem a single LLM call cannot solve |
| 2. The four pillars of agentic design | What makes a program an *agent* |
| 3. The patterns | One diagram + one small example per pattern |
| 4. Picking a pattern | A cheat-sheet to choose quickly |

> Diagrams are Mermaid blocks, they render directly in VS Code / Jupyter markdown cells.

---

## 1. Why agentic design patterns

A plain LLM call is a **one-shot function**: prompt in, text out. It cannot look anything up,
cannot check its own work, and forgets everything the moment it answers.

An agent wraps the same model in a **loop** with tools, memory and a goal, so it can keep
working until the goal is met.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    subgraph ONE["One-shot LLM"]
        direction LR
        P1(["Prompt"]) --> M1["LLM call"] --> A1(["Answer"])
    end

    subgraph AGENT["Agent loop"]
        direction LR
        G2(["Goal"]) --> T2["Decide next action"]
        T2 --> AC2["Use tools or agents"]
        AC2 --> O2["Observe result"]
        O2 --> CHECK2{"Goal met?"}
        CHECK2 -->|"No"| T2
        CHECK2 -->|"Yes"| D2(["Final answer"])
    end

    ONE -.->|"add tools, state and a loop"| AGENT

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class P1,G2 source
    class M1,T2,AC2,O2 process
    class CHECK2 decision
    class A1,D2 success
```

**The recurring problems, and the pattern that solves each**

| Pain with a single LLM call | Pattern that fixes it |
| --- | --- |
| Task is too big for one prompt | Prompt Chaining, Planning |
| Answer is confidently wrong | Reflection |
| Model has no live or private data | Tool Use |
| Every request handled the same way | Routing |
| Sequential work is slow | Parallelization |
| Forgets the previous turn | Memory |
| One prompt juggling many skills | Multi-Agent Collaboration |
| Two plausible answers, no way to choose | Group Chat / Debate (a multi-agent variant) |
| Risky or irreversible actions | Human-in-the-Loop |

**Patterns are not frameworks.** They are repeatable shapes of control flow, in LangGraph each
one is just a small arrangement of *nodes* (steps) and *edges* (what runs next).

---

## 2. The four pillars of agentic design

These four properties are what separate an AI agent from a traditional program.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart TB
    G["🎯 <b>Goal-Directed</b><br/>driven by an objective,<br/>not just an instruction"]
    R["👀 <b>Reactive</b><br/>perceives and responds<br/>to its environment"]
    AGENT["🤖 <b>AI Agent</b>"]
    AU["🚀 <b>Autonomous</b><br/>acts on its own to make<br/>progress toward the goal"]
    S["🧠 <b>Stateful</b><br/>keeps memory of past<br/>events and actions"]

    G --- AGENT
    R --- AGENT
    AGENT --- AU
    AGENT --- S

    classDef goal fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef reactive fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef core fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:3px
    classDef autonomous fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    classDef stateful fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    class G goal
    class R reactive
    class AGENT core
    class AU autonomous
    class S stateful
```

| Pillar | Plain program | Agent | Where it shows up in LangGraph |
| --- | --- | --- | --- |
| **Goal-Directed** | Runs fixed steps | Keeps working until the objective is met | The loop back into the model node |
| **Reactive** | Ignores the world | Reads tool results, errors, user replies | Tool node output fed back as messages |
| **Stateful** | Forgets each call | Remembers the conversation and what it tried | Graph `state` + checkpointer |
| **Autonomous** | Waits to be told | Decides the next action itself | Conditional edges chosen by the model |

**One-line test:** if you can replace it with an `if/else` and a template, it is not an agent.

---

## 3. The patterns

Each pattern below is: **one idea → one diagram → one tiny example**.
Every code cell is runnable and deliberately tiny - the goal is to see the shape of the pattern, not production code.

### Setup - run this first

Every example below reuses one chat model and one tiny helper, `ask()`, so each pattern cell stays
short enough to read in one go. Install the dependencies, add your Azure OpenAI configuration to a local
`.env` file, then run the setup cell. Configuration is read at runtime; no credential values are embedded here.

```bash
pip install -U langchain langchain-openai langgraph python-dotenv
```

Required environment variables: `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT_NAME`, and
`AZURE_OPENAI_API_KEY`. Keep the `.env` file local and out of source control.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    ENV(["Local environment<br/>configuration"]) --> LOAD["load_dotenv()"]
    LOAD --> LLM["init_chat_model(...)"]
    LLM --> ASK["ask(prompt) -> text"]
    ASK --> PAT(["Nine runnable patterns"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class ENV source
    class LOAD,LLM,ASK process
    class PAT success
```

In [2]:
import os
import time
from typing import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")

missing = [
    name
    for name, value in {
        "AZURE_OPENAI_ENDPOINT": ENDPOINT,
        "AZURE_OPENAI_DEPLOYMENT_NAME": DEPLOYMENT,
        "AZURE_OPENAI_API_KEY": API_KEY,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Set these environment variables before continuing: {', '.join(missing)}")

# The OpenAI-compatible Foundry surface lives under /openai/v1, so add it if .env holds the
# bare resource URL. For a classic Azure OpenAI resource you would instead use:
#   init_chat_model(f"azure_openai:{DEPLOYMENT}", azure_endpoint=ENDPOINT, api_version=...)
BASE_URL = ENDPOINT if ENDPOINT.endswith("/openai/v1") else f"{ENDPOINT}/openai/v1"

llm = init_chat_model(
    f"openai:{DEPLOYMENT}",
    base_url=BASE_URL,
    api_key=API_KEY,
)


def ask(prompt: str) -> str:
    """Send one prompt, get plain text back. Keeps the pattern examples tiny."""
    return str(llm.invoke(prompt).content).strip()


print("Model ready:", DEPLOYMENT)
print("Smoke test :", ask("Reply with exactly: ready"))

Model ready: gpt-5.4
Smoke test : ready


---

### 3.1 Prompt Chaining (Sequential)

**Idea:** Break one big task into small steps, and feed each step's output into the next.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    IN(["Support email"]) --> S1["extract()<br/>Find core issue"]
    S1 --> S2["summarize()<br/>At most 10 words"]
    S2 --> S3["make_ticket()<br/>TITLE | PRIORITY"]
    S3 --> OUT(["Ticket line"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class IN source
    class S1,S2,S3 process
    class OUT success
```

**Example:** a support email → *extract* the customer issue → *summarize* it in under 10 words → *format* it as a ticket line.

**In LangGraph:** one node per step, joined by plain edges. No branching, no loop.

In [20]:
# Pattern: one node per step, joined in a straight line.
EMAIL = (
    "Hi, I was charged twice for order #4471 last Friday and nobody replied "
    "to my two emails. Please refund the extra charge."
)


class ChainState(TypedDict):
    email: str
    issue: str
    summary: str
    ticket: str


def extract(state: ChainState) -> dict:
    return {"issue": ask(f"Extract the customer's core problem in one line:\n{state['email']}")}


def summarize(state: ChainState) -> dict:
    return {"summary": ask(f"Rewrite in at most 10 words:\n{state['issue']}")}


def make_ticket(state: ChainState) -> dict:
    return {
        "ticket": ask(
            "Format as 'TITLE | PRIORITY' where priority is low, medium or high. "
            f"Reply with that line only:\n{state['summary']}"
        )
    }


chain = (
    StateGraph(ChainState)
    .add_node("extract", extract)
    .add_node("summarize", summarize)
    .add_node("make_ticket", make_ticket)
    
    .add_edge(START, "extract")
    .add_edge("extract", "summarize")
    .add_edge("summarize", "make_ticket")
    .add_edge("make_ticket", END)
    .compile()
)

result = chain.invoke({"email": EMAIL})
for step in ("issue", "summary", "ticket"):
    print(f"{step:8}: {result[step]}")

issue   : Customer was double-charged for order #4471 last Friday and is requesting a refund for the duplicate charge after receiving no response to two emails.
summary : Customer seeks duplicate-charge refund for order #4471 after no response.
ticket  : Duplicate-charge refund request for order #4471 | high


---

### 3.2 Routing

**Idea:** Classify the request first, then send it down the branch that is built for it.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    Q(["User request"]) --> R{"classify()<br/>billing, technical or other?"}
    R -->|"billing"| B["Billing agent"]
    R -->|"technical"| T["Technical support engineer"]
    R -->|"other"| F["General assistant"]
    B --> OUT(["One-sentence answer"])
    T --> OUT
    F --> OUT

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class Q source
    class R decision
    class B,T,F process
    class OUT success
```

**Example:** "My card was charged twice" → router picks **billing**, skipping the tech-support tools entirely.

**In LangGraph:** a conditional edge, the router node returns the name of the next node.

In [21]:
# Pattern: classify first, then take exactly one branch.
class RouteState(TypedDict):
    question: str
    route: str
    answer: str


def classify(state: RouteState) -> dict:
    choice = ask(
        "Classify the message. Reply with exactly one word - billing, technical or other:\n"
        f"{state['question']}"
    ).lower().strip(".")
    route = choice if choice in {"billing", "technical"} else "other"
    print(f"  router -> {route}")
    return {"route": route}


def answer_as(role: str):
    def node(state: RouteState) -> dict:
        return {"answer": ask(f"You are a {role}. Answer in one sentence:\n{state['question']}")}

    return node


router = (
    StateGraph(RouteState)
    .add_node("classify", classify)
    .add_node("billing", answer_as("billing agent"))
    .add_node("technical", answer_as("technical support engineer"))
    .add_node("other", answer_as("general assistant"))
    .add_edge(START, "classify")
    # The conditional edge picks the next node using the value the router wrote into state.
    .add_conditional_edges("classify", lambda s: s["route"], ["billing", "technical", "other"])
    .add_edge("billing", END)
    .add_edge("technical", END)
    .add_edge("other", END)
    .compile()
)

for question in [
    "I was charged twice this month.",
    "The app crashes when I upload a photo.",
    "How are you doing today."
]:
    print(question)
    print("  answer ->", router.invoke({"question": question})["answer"], "\n")

I was charged twice this month.
  router -> billing
  answer -> I’m sorry about that—please share the charge dates, amounts, and the last four digits of the card used, and I’ll help investigate the duplicate charge. 

The app crashes when I upload a photo.
  router -> technical
  answer -> Please update the app to the latest version, clear its cache, restart your device, and if it still crashes, reinstall the app and send us your device model, OS version, app version, and a screenshot or screen recording of the issue. 

How are you doing today.
  router -> other
  answer -> I’m doing well, thank you—how are you today? 



---

### 3.3 Parallelization

**Idea:** Run independent sub-tasks at the same time, then merge the results.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    Q(["Topic:<br/>coffee vs tea"]) --> A["pro()<br/>Find a benefit"]
    Q --> B["con()<br/>Find a drawback"]
    A --> J["verdict()<br/>Wait for both"]
    B --> J
    J --> OUT(["Balanced verdict"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef parallel fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class Q source
    class A,B parallel
    class J process
    class OUT success
```

**Example:** "Is coffee better than tea?" - the pro and con are written at the same time, then one node joins them into a verdict. Watch the timestamps in the output: both branches start at t+0.0s.

**In LangGraph:** two edges leaving `START` create branches that run together, and the node they both feed into waits for both results before producing the verdict.

In [22]:
# Pattern: two branches leave START together, then join in one node.
class ReviewState(TypedDict):
    topic: str
    pro: str
    con: str
    verdict: str


def timed(label: str, prompt_template: str, key: str):
    def node(state: ReviewState) -> dict:
        started = time.perf_counter() - T0
        text = ask(prompt_template.format(topic=state["topic"]))
        # Both nodes start at ~the same second: that overlap is the whole point.
        print(f"  {label} ran from t+{started:.1f}s to t+{time.perf_counter() - T0:.1f}s")
        return {key: text}

    return node


parallel = (
    StateGraph(ReviewState)
    .add_node("pro", timed("pro    ", "One short benefit of {topic}. One sentence.", "pro"))
    .add_node("con", timed("con    ", "One short drawback of {topic}. One sentence.", "con"))
    .add_node(
        "verdict",
        lambda s: {"verdict": ask(f"Pro: {s['pro']}\nCon: {s['con']}\nOne-sentence verdict.")},
    )
    # Both edges leave START, so LangGraph runs the two nodes at the same time.
    .add_edge(START, "pro")
    .add_edge(START, "con")
    .add_edge("pro", "verdict")
    .add_edge("con", "verdict")
    .add_edge("verdict", END)
    .compile()
)

T0 = time.perf_counter()
out = parallel.invoke({"topic": "Is coffee better than tea"})
print(f"  verdict ran after the join, total t+{time.perf_counter() - T0:.1f}s\n")
print("pro    :", out["pro"])
print("con    :", out["con"])
print("verdict:", out["verdict"])

  pro     ran from t+0.0s to t+3.3s
  con     ran from t+0.0s to t+3.8s
  verdict ran after the join, total t+5.5s

pro    : Coffee often provides a quicker, stronger energy boost than tea due to its higher caffeine content.
con    : A short drawback of coffee compared to tea is that it often contains more caffeine, which can cause jitters or disrupt sleep.
verdict: Coffee gives a stronger, faster energy boost than tea, but its higher caffeine content can also lead to jitters or sleep disruption, so the better choice depends on your tolerance and needs.


---

### 3.4 Reflection (Self-Critique)

**Idea:** The agent reviews its own draft against the goal and revises until it is good enough.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    T(["Reusable-bottle<br/>tagline task"]) --> GEN["generate()<br/>Draft tagline"]
    GEN --> CRIT{"critique()<br/>All three rules pass?"}
    CRIT -->|"Fix requested<br/>and rounds remain"| GEN
    CRIT -->|"APPROVED<br/>or round cap"| OUT(["Final tagline"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class T source
    class GEN process
    class CRIT decision
    class OUT success
```

**Example:** write a tagline for a reusable water bottle against three rules (under 8 words, mentions reuse, no exclamation marks). The critic replies `APPROVED` or one fix instruction, and the generator tries again. If the very first draft happens to pass, the loop exits after one round - re-run the cell to watch it revise.

**In LangGraph:** a generate node and a critique node with a conditional edge looping back, plus a max-iteration guard.

In [24]:
# Pattern: generate -> critique -> loop back, with a hard round cap.
MAX_ROUNDS = 3
RULES = "under 8 words, mentions reuse, no exclamation marks"


class DraftState(TypedDict):
    task: str
    draft: str
    feedback: str
    round: int


def generate(state: DraftState) -> dict:
    prompt = f"Task: {state['task']}\nReply with the tagline only."
    if state["feedback"]:
        prompt += f"\nYour previous try: {state['draft']}\nFix this: {state['feedback']}"
    draft = ask(prompt)
    round_no = state["round"] + 1
    print(f"  round {round_no} draft    : {draft}")
    return {"draft": draft, "round": round_no}


def critique(state: DraftState) -> dict:
    feedback = ask(
        f"You are a strict editor. Rules: {RULES}.\n"
        f"Tagline: {state['draft']}\n"
        "Reply APPROVED if every rule passes, otherwise one short fix instruction."
    )
    print(f"  round {state['round']} critique : {feedback}")
    return {"feedback": feedback}


def keep_going(state: DraftState) -> str:
    if state["feedback"].upper().startswith("APPROVED"):
        return END
    if state["round"] >= MAX_ROUNDS:
        print("  round cap reached, stopping")
        return END
    return "generate"


reflect = (
    StateGraph(DraftState)
    .add_node("generate", generate)
    .add_node("critique", critique)
    .add_edge(START, "generate")
    .add_edge("generate", "critique")
    .add_conditional_edges("critique", keep_going, ["generate", END])
    .compile()
)

final = reflect.invoke(
    {
        "task": "Write a tagline for a reusable water bottle.",
        "draft": "",
        "feedback": "",
        "round": 0,
    }
)
print("\nfinal tagline:", final["draft"])

  round 1 draft    : Refill Your Day.
  round 1 critique : Add reuse mention.
  round 2 draft    : Refill. Reuse. Repeat.
  round 2 critique : APPROVED

final tagline: Refill. Reuse. Repeat.


---

### 3.5 Tool Use (Function Calling)

**Idea:** Give the model typed functions so it can fetch real data or cause real effects.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    U(["User question"]) --> M{"Model<br/>call a tool?"}
    M -->|"Weather request"| TOOL["get_weather(city)"]
    TOOL -->|"Tool result"| M
    M -->|"No tool needed"| OUT(["Final answer"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef tool fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class U source
    class M decision
    class TOOL tool
    class OUT success
```

**Example:** "Weather in Paris today?" → the model calls `get_weather("Paris")` and answers from the returned value instead of guessing.

**In LangGraph:** `create_agent(..., tools=[...])`, the model↔tool hop repeats until no tool is requested.

In [11]:
# Pattern: the model decides when to call a function, and answers from its result.
from random import randint

from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get today's weather for a city."""
    print(f"  [tool called] get_weather({city!r})")
    return f"{city}: sunny, {randint(15, 30)}C"


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="Use the get_weather tool for weather questions. Answer in one sentence.",
)

reply = weather_agent.invoke({"messages": [{"role": "user", "content": "Weather in Paris?"}]})
print("answer:", reply["messages"][-1].content)

# No tool is needed here, so the model skips it - that decision is the pattern.
reply = weather_agent.invoke({"messages": [{"role": "user", "content": "What is 2 + 2?"}]})
print("answer:", reply["messages"][-1].content)

  [tool called] get_weather('Paris')
answer: Paris is sunny and 15°C today.
answer: 2 + 2 = 4.


---

### 3.6 Planning

**Idea:** Turn a vague goal into an explicit list of steps first, then execute the steps.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    G(["Goal:<br/>one day in Goa"]) --> P["planner()<br/>Create 3 steps"]
    P --> PLAN["steps[]<br/>Shared plan state"]
    PLAN --> EX["executor()<br/>Complete next step"]
    EX --> MORE{"more_steps()<br/>Steps left?"}
    MORE -->|"Yes"| EX
    MORE -->|"No"| OUT(["Completed itinerary"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef state fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class G source
    class P,EX process
    class PLAN state
    class MORE decision
    class OUT success
```

**Example:** "Plan a one-day trip to Goa" - the planner writes 3 short steps, then the executor loops over them one at a time, filling in a concrete recommendation for each.

**In LangGraph:** a planner node writes a step list into state, then an executor node completes one step at a time and loops until the list is exhausted.

In [32]:
# Pattern: write the steps first, then execute them one at a time.
class PlanState(TypedDict):
    goal: str
    steps: list[str]
    done: list[str]


def planner(state: PlanState) -> dict:
    raw = ask(
        f"Goal: {state['goal']}\n"
        "List exactly 3 steps, at most 6 words each, one per line, no numbering."
    )
    steps = [line.strip("-*0123456789. ") for line in raw.splitlines() if line.strip()][:3]
    print("  plan:")
    for i, step in enumerate(steps, 1):
        print(f"    {i}. {step}")
    return {"steps": steps, "done": []}


def executor(state: PlanState) -> dict:
    step = state["steps"][len(state["done"])]
    result = ask(
        f"Goal: {state['goal']}\nStep: {step}\n"
        "Give one concrete recommendation for this step only, in one sentence."
    )
    print(f"  step {len(state['done']) + 1} -> {result}")
    return {"done": state["done"] + [result]}


def more_steps(state: PlanState) -> str:
    # Loop back into the executor until every planned step is done.
    return "executor" if len(state["done"]) < len(state["steps"]) else END


planning = (
    StateGraph(PlanState)
    .add_node("planner", planner)
    .add_node("executor", executor)
    .add_edge(START, "planner")
    .add_edge("planner", "executor")
    .add_conditional_edges("executor", more_steps, ["executor", END])
    .compile()
)

planning.invoke({"goal": "Plan a simple one-day trip to Goa.", "steps": [], "done": []})

  plan:
    1. Arrive early at Baga Beach
    2. Explore Fort Aguada by noon
    3. Enjoy sunset and seafood dinner
  step 1 -> Reach Baga Beach by 7:00 AM to enjoy the quiet shoreline, cooler weather, and easier parking before the crowds arrive.
  step 2 -> Reach Fort Aguada by 10:00 AM and spend about 90 minutes exploring the lighthouse, ramparts, and Arabian Sea viewpoints before the midday heat sets in.
  step 3 -> Head to **Fisherman’s Wharf in Cavelossim** for a sunset-side seafood dinner with Goan specialties like prawn curry and grilled fish.


{'goal': 'Plan a simple one-day trip to Goa.',
 'steps': ['Arrive early at Baga Beach',
  'Explore Fort Aguada by noon',
  'Enjoy sunset and seafood dinner'],
 'done': ['Reach Baga Beach by 7:00 AM to enjoy the quiet shoreline, cooler weather, and easier parking before the crowds arrive.',
  'Reach Fort Aguada by 10:00 AM and spend about 90 minutes exploring the lighthouse, ramparts, and Arabian Sea viewpoints before the midday heat sets in.',
  'Head to **Fisherman’s Wharf in Cavelossim** for a sunset-side seafood dinner with Goan specialties like prawn curry and grilled fish.']}

---

### 3.7 Multi-Agent Collaboration

**Idea:** Split the work across specialists and let a supervisor coordinate them.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart TB
    U(["Write 2 sentences about<br/>the Great Rift Valley"]) --> SUP{"Supervisor<br/>who acts next?"}
    SUP -->|"researcher"| RES["Researcher<br/>Collect 2 facts"]
    SUP -->|"writer"| WRI["Writer<br/>Create the draft"]
    SUP -->|"reviewer"| REV["Reviewer<br/>Return OK or one fix"]
    RES --> SUP
    WRI --> SUP
    REV --> SUP
    SUP -->|"done"| OUT(["Final draft"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef agent fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class U source
    class SUP decision
    class RES,WRI,REV agent
    class OUT success
```

**Example:** two sentences about the Great Rift Valley - the researcher gathers facts, the writer drafts, the reviewer checks it, and the supervisor decides who acts next until the work is complete.

**In LangGraph:** each agent is a node, the supervisor node routes with a conditional edge and shared state carries the work product.

**Variant: Group Chat / Debate.** Same cast of agents, different topology - instead of a supervisor
handing out *different* sub-tasks, the agents argue *the same* question in one shared transcript and a
judge decides.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    Q(["Postgres or DynamoDB<br/>for 100 users?"]) --> A["Agent A<br/>Argue for Postgres"]
    A --> T1["Append to<br/>shared transcript"]
    T1 --> B["Agent B<br/>Argue for DynamoDB"]
    B --> T2["Append to<br/>shared transcript"]
    T2 --> MOD{"Two rounds<br/>complete?"}
    MOD -->|"No"| A
    MOD -->|"Yes"| J["Judge<br/>Weigh all arguments"]
    J --> OUT(["Decision + trade-off"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef agent fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef state fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class Q source
    class A,B,J agent
    class T1,T2 state
    class MOD decision
    class OUT success
```

| | Supervisor (above) | Group Chat / Debate |
| --- | --- | --- |
| Agents work on | **Different** sub-tasks | The **same** question |
| Shape | Hub and spoke | Everyone sees everyone |
| Good for | Dividing labour | High-stakes calls with no obvious winner |
| Stops when | The work items are done | Agents converge, or the round cap is hit |

**Example:** "Postgres or DynamoDB for this service?" - round 1 each agent makes its case, round 2 each
rebuts the other's claims, then the judge picks a winner and states the trade-off it accepted.

In [27]:
# Pattern: specialists do different sub-tasks, a supervisor picks who acts next.
class TeamState(TypedDict):
    topic: str
    notes: str
    draft: str
    review: str
    next: str


def supervisor(state: TeamState) -> dict:
    status = (
        f"notes={'yes' if state['notes'] else 'no'}, "
        f"draft={'yes' if state['draft'] else 'no'}, "
        f"review={'yes' if state['review'] else 'no'}"
    )
    choice = ask(
        "You coordinate three workers: researcher (gathers facts), writer (writes the draft), "
        f"reviewer (checks the draft). Work done so far: {status}. "
        "Reply with exactly one word - researcher, writer, reviewer or done."
    ).lower().strip(".")
    print(f"  supervisor -> {choice}")
    return {"next": choice}


def researcher(state: TeamState) -> dict:
    notes = ask(f"List 2 short facts about {state['topic']}.")
    print(f"  researcher : {notes.splitlines()[0]} ...")
    return {"notes": notes}


def writer(state: TeamState) -> dict:
    draft = ask(f"Write 2 sentences about {state['topic']} using these facts:\n{state['notes']}")
    print(f"  writer     : {draft}")
    return {"draft": draft}


def reviewer(state: TeamState) -> dict:
    review = ask(f"Reply OK, or one short fix, for this text:\n{state['draft']}")
    print(f"  reviewer   : {review}")
    return {"review": review}


def route(state: TeamState) -> str:
    # Safety net: stop once every worker has contributed, whatever the supervisor says.
    if state["notes"] and state["draft"] and state["review"]:
        return END
    return state["next"] if state["next"] in {"researcher", "writer", "reviewer"} else END


team = (
    StateGraph(TeamState)
    .add_node("supervisor", supervisor)
    .add_node("researcher", researcher)
    .add_node("writer", writer)
    .add_node("reviewer", reviewer)
    .add_edge(START, "supervisor")
    .add_conditional_edges("supervisor", route, ["researcher", "writer", "reviewer", END])
    .add_edge("researcher", "supervisor")
    .add_edge("writer", "supervisor")
    .add_edge("reviewer", "supervisor")
    .compile()
)

done = team.invoke({"topic": "the Great Rift Valley", "notes": "", "draft": "", "review": "", "next": ""})
print("\nfinal draft:", done["draft"])

  supervisor -> researcher
  researcher : - The Great Rift Valley is a huge geological trench stretching from the Middle East through eastern Africa.   ...
  supervisor -> writer
  writer     : The Great Rift Valley is a massive geological trench that stretches from the Middle East through eastern Africa. It formed as tectonic plates pulled apart, creating deep valleys, volcanoes, and lakes along the way.
  supervisor -> reviewer
  reviewer   : OK
  supervisor -> done

final draft: The Great Rift Valley is a massive geological trench that stretches from the Middle East through eastern Africa. It formed as tectonic plates pulled apart, creating deep valleys, volcanoes, and lakes along the way.


In [28]:
# Variant: same agents, but they argue the SAME question in one shared transcript.
# Written as a plain loop so the round structure is obvious.
QUESTION = "For a hobby app with 100 users, is Postgres or DynamoDB the better first database?"
ROUNDS = 2
SIDES = [("Agent A", "Postgres"), ("Agent B", "DynamoDB")]

transcript: list[str] = []

for round_no in range(1, ROUNDS + 1):
    print(f"-- round {round_no} --")
    for name, side in SIDES:
        so_far = "\n".join(transcript) or "(nothing yet)"
        turn = ask(
            f"Question: {QUESTION}\n"
            f"You argue for {side}. Transcript so far:\n{so_far}\n"
            "Reply with one short sentence, rebutting the other side if it has spoken."
        )
        transcript.append(f"{name} ({side}): {turn}")
        print(" ", transcript[-1])

verdict = ask(
    f"Question: {QUESTION}\nDebate:\n" + "\n".join(transcript) + "\n"
    "You are the judge. Reply as '<winner> - <one line reason>'."
)
print("\njudge:", verdict)

-- round 1 --
  Agent A (Postgres): Postgres is the better first choice for a 100-user hobby app because it’s simpler to build with, cheaper to reason about, and avoids DynamoDB’s upfront data-modeling complexity.
  Agent B (DynamoDB): DynamoDB is the better first database because for a tiny hobby app it gives you zero-ops reliability and effortless scaling, while Postgres still makes you own migrations, tuning, and failure modes.
-- round 2 --
  Agent A (Postgres): At 100 users, DynamoDB’s “effortless scaling” solves a problem you don’t have, while Postgres is far easier to model, query, and iterate with.
  Agent B (DynamoDB): For a hobby app, avoiding even small operational burdens matters more than hypothetical query flexibility, and DynamoDB lets you ship without ever babysitting the database.

judge: Postgres - For a 100-user hobby app, ease of modeling and iteration matters more than DynamoDB’s premature scaling benefits.


---

### 3.8 Memory

**Idea:** Preserve conversation context across turns while keeping each thread isolated.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    A1(["Alice turn 1<br/>I am vegetarian"]) --> AAG["Agent<br/>thread_id = alice"]
    AAG <--> MEM["InMemorySaver<br/>Alice's messages"]
    A2(["Alice turn 2<br/>Suggest dinner"]) --> AAG
    AAG --> AOUT(["Vegetarian suggestion"])

    B1(["Bob<br/>Suggest dinner"]) --> BAG["Same agent<br/>thread_id = bob"]
    BAG --> BOUT(["No saved preference"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef memory fill:#FAE8FF,stroke:#A21CAF,color:#701A75,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class A1,A2,B1 source
    class AAG,BAG process
    class MEM memory
    class AOUT,BOUT success
```

**Example:** turn 1 - "I am vegetarian", a later turn - "suggest dinner" → the agent still avoids meat. The same question on a different `thread_id` gets a meat dish, because that thread never heard it.

**In LangGraph:** message state plus a checkpointer preserves context for each `thread_id`. This example intentionally demonstrates short-term, thread-scoped memory only.

In [30]:
# Pattern: a checkpointer plus a thread_id turns a stateless agent into a remembering one.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Answer in one short sentence.",
    checkpointer=InMemorySaver(),
)

alice = {"configurable": {"thread_id": "alice"}}
bob = {"configurable": {"thread_id": "bob"}}

memory_agent.invoke({"messages": [{"role": "user", "content": "Remember: I am vegetarian."}]}, alice)

same = memory_agent.invoke({"messages": [{"role": "user", "content": "Suggest one dinner."}]}, alice)
print("alice (remembers):", same["messages"][-1].content)

other = memory_agent.invoke({"messages": [{"role": "user", "content": "Suggest one dinner."}]}, bob)
print("bob   (no memory):", other["messages"][-1].content)

alice (remembers): How about a chickpea and spinach curry with basmati rice?
bob   (no memory): Try baked salmon with roasted asparagus and quinoa.


---

### 3.9 Human-in-the-Loop

**Idea:** Pause before risky or irreversible actions and wait for a human decision.

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart LR
    START0(["Refund request<br/>$4,000"]) --> H["ask_human()<br/>interrupt() pauses"]
    H --> INPUT{"Human input"}
    INPUT -->|"approve"| RUN["execute()<br/>Send refund"]
    INPUT -->|"reject"| STOP["execute()<br/>Skip refund"]
    RUN --> OUT(["status = sent"])
    STOP --> OUT2(["status = skipped"])

    classDef source fill:#E0F2FE,stroke:#0284C7,color:#0C4A6E,stroke-width:2px
    classDef human fill:#FEF3C7,stroke:#D97706,color:#78350F,stroke-width:2px
    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef process fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef success fill:#ECFDF5,stroke:#059669,color:#064E3B,stroke-width:2px
    class START0 source
    class H human
    class INPUT decision
    class RUN,STOP process
    class OUT,OUT2 success
```

**Example:** the agent is about to send a refund of $4,000 → the run pauses and asks → enter `approve` or `reject` when prompted to resume it.

**In LangGraph:** an interrupt before the tool node, the graph resumes from the checkpoint with the human's answer.

In [31]:
# Pattern: interrupt() pauses the graph mid-run, Command(resume=...) continues it.
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class RefundState(TypedDict):
    amount: int
    status: str


def ask_human(state: RefundState) -> dict:
    decision = interrupt(f"Approve a refund of ${state['amount']}? (approve / reject)")
    return {"status": decision}


def execute(state: RefundState) -> dict:
    if state["status"] == "approve":
        print(f"  refund of ${state['amount']} sent")
        return {"status": "sent"}
    print("  refund skipped")
    return {"status": "skipped"}


refund = (
    StateGraph(RefundState)
    .add_node("ask_human", ask_human)
    .add_node("execute", execute)
    .add_edge(START, "ask_human")
    .add_edge("ask_human", "execute")
    .add_edge("execute", END)
    # A checkpointer is required: the paused run is stored and resumed from it.
    .compile(checkpointer=InMemorySaver())
)

thread_id = "refund-1"
config = {"configurable": {"thread_id": thread_id}}
paused = refund.invoke({"amount": 4000, "status": ""}, config)
question = paused["__interrupt__"][0].value

while True:
    decision = input(f"{question} ").strip().lower()
    if decision in {"approve", "reject"}:
        break
    print("Please enter 'approve' or 'reject'.")

final = refund.invoke(Command(resume=decision), config)
print(f"{thread_id} resumed with {decision!r} -> status = {final['status']}")

  refund of $4000 sent
refund-1 resumed with 'approve' -> status = sent


---

## 4. Picking a pattern

```mermaid
%%{init: {"theme": "base", "themeVariables": {"fontFamily": "Inter, Segoe UI, sans-serif", "primaryTextColor": "#0F172A", "lineColor": "#64748B"}}}%%
flowchart TB
    Q{"What failure<br/>must you remove?"}
    Q -->|"Task too big"| P1["Prompt Chaining<br/>or Planning"]
    Q -->|"Many request types"| P2["Routing"]
    Q -->|"Independent work is slow"| P3["Parallelization"]
    Q -->|"Quality is inconsistent"| P4["Reflection"]
    Q -->|"Needs external data"| P5["Tool Use"]
    Q -->|"One prompt has too many roles"| P6["Multi-Agent<br/>Supervisor"]
    Q -->|"No obvious winner"| P9["Multi-Agent<br/>Debate"]
    Q -->|"Forgets context"| P7["Memory"]
    Q -->|"Action needs approval"| P8["Human-in-the-Loop"]

    classDef decision fill:#FFF7ED,stroke:#EA580C,color:#7C2D12,stroke-width:3px
    classDef pattern fill:#EEF2FF,stroke:#4F46E5,color:#312E81,stroke-width:2px
    classDef safety fill:#FEF3C7,stroke:#D97706,color:#78350F,stroke-width:2px
    class Q decision
    class P1,P2,P3,P4,P5,P6,P9,P7 pattern
    class P8 safety
```

**Rules of thumb**

1. Start with the simplest thing that works: a single prompt, then Tool Use, then a loop.
2. Add a pattern only when you can name the failure it removes.
3. Patterns compose - a real agent is usually *Routing + Tool Use + Memory*, with Reflection on the parts that must be correct.
4. Every loop needs a stop condition: a max iteration count, a budget, or a human.

---

### Where to go next

Try combining two patterns in one graph - for example Routing + Tool Use or Planning + Reflection. Then add tracing around each node, compare the path taken with the diagram, and share what changed when the model had tools, memory, or human oversight.

> **Your turn:** Which pattern would remove the biggest failure mode in your current AI workflow?